# 1. 라이브러리 로드

In [1]:
!pip install koreanize-matplotlib transformers

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd

import math

import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

import warnings
warnings.filterwarnings('ignore')

# 2. 데이터 로드

In [3]:
col_to_use = [
    'region', 'date', 'volume_cnt',
    'avg_area_size', 'avg_sales', 'avg_floor_cnt',

    'pod_index', 'abs_vdd_index', 'canceld_index', 'direct_trade_ratio', 'anomaly_abs_vdd_score',

    'wolse_volume_cnt', 'jeonse_volume_cnt', 'wolse_avg_area_size',
    'jeonse_avg_area_size', 'wolse_avg_deposit', 'jeonse_avg_deposit',
    'wolse_avg_monthly', 'wolse_avg_floor_cnt', 'jeonse_avg_floor_cnt',

    'townhouse_volume_cnt', 'townhouse_avg_area_size',
    'townhouse_avg_sales', 'townhouse_avg_floor_cnt',
    'townhouse_jeonse_volume_cnt', 'townhouse_wolse_avg_area_size',
    'townhouse_jeonse_avg_area_size', 'townhouse_wolse_avg_floor_cnt',
    'townhouse_jeonse_avg_floor_cnt', 'townhouse_wolse_avg_deposit',
    'townhouse_jeonse_avg_deposit', 'townhouse_wolse_avg_monthly',

    'familyhouse_volume_cnt', 'familyhouse_avg_area_size',
    'familyhouse_avg_sales', 'familyhouse_wolse_volume_cnt',
    'familyhouse_jeonse_volume_cnt', 'familyhouse_wolse_avg_area_size',
    'familyhouse_jeonse_avg_area_size', 'familyhouse_wolse_avg_deposit',
    'familyhouse_jeonse_avg_deposit', 'familyhouse_wolse_avg_monthly',

    'officehouse_volume_cnt', 'officehouse_avg_area_size',
    'officehouse_avg_sales', 'officehouse_avg_floor_cnt',
    'officehouse_wolse_volume_cnt', 'officehouse_jeonse_volume_cnt',
    'officehouse_wolse_avg_area_size', 'officehouse_jeonse_avg_area_size',
    'officehouse_wolse_avg_floor_cnt', 'officehouse_jeonse_avg_floor_cnt',
    'officehouse_wolse_avg_deposit', 'officehouse_jeonse_avg_deposit',
    'officehouse_wolse_avg_monthly', 'townhouse_wolse_volume_cnt',

    'is_anomaly_score_outlier_mean_1sigma_win5',
       'is_anomaly_score_outlier_mean_2sigma_win5',
       'is_anomaly_score_outlier_mean_3sigma_win5',
       'is_anomaly_score_outlier_mean_1sigma_win10',
       'is_anomaly_score_outlier_mean_2sigma_win10',
       'is_anomaly_score_outlier_mean_3sigma_win10',
       'is_anomaly_score_outlier_mean_1sigma_win15',
       'is_anomaly_score_outlier_mean_2sigma_win15',
       'is_anomaly_score_outlier_mean_3sigma_win15',
       'is_anomaly_score_outlier_mean_1sigma_win20',
       'is_anomaly_score_outlier_mean_2sigma_win20',
       'is_anomaly_score_outlier_mean_3sigma_win20',
       'is_anomaly_score_outlier_mean_1sigma_win25',
       'is_anomaly_score_outlier_mean_2sigma_win25',
       'is_anomaly_score_outlier_mean_3sigma_win25',
       'is_anomaly_score_outlier_mean_1sigma_win30',
       'is_anomaly_score_outlier_mean_2sigma_win30',
       'is_anomaly_score_outlier_mean_3sigma_win30'
]

In [4]:
apt_volume_train_df = pd.read_csv('/content/drive/MyDrive/sogang_thesis_prj/dataset/final_test3/apt_sales_volume_daily_register_202111_202412_train_anomaly_score_label_combi3.csv')[col_to_use]
apt_volume_valid_df = pd.read_csv('/content/drive/MyDrive/sogang_thesis_prj/dataset/final_test3/apt_sales_volume_daily_register_202111_202412_valid_anomaly_score_label_combi3.csv')[col_to_use]
apt_volume_test_df = pd.read_csv('/content/drive/MyDrive/sogang_thesis_prj/dataset/final_test3/apt_sales_volume_daily_register_202111_202412_test_anomaly_score_label_combi3.csv')[col_to_use]

In [5]:
len(apt_volume_train_df), len(apt_volume_valid_df), len(apt_volume_test_df)

(20550, 3025, 5350)

# 3. 데이터 전처리

In [6]:
def scale_features_from_split(train_df, valid_df, test_df, features, target_col=None, target_clean_col=None):
    """
    train_df 기준으로 MinMaxScaler fit 후 전체 transform.
    - train/valid: 보정된 종속변수 target_clean_col 기준으로 transform
    - test: 원본 종속변수 target_col 기준으로 transform
    - 반환되는 결과는 모두 DataFrame
    """

    # 👉 fit 시 사용할 feature + 보정된 target 컬럼
    train_df_copy = train_df.copy()[features + [target_col]].rename(columns={target_col: 'target'})
    valid_df_copy = valid_df.copy()[features + [target_col]].rename(columns={target_col: 'target'})
    test_df_copy  = test_df.copy()[features + [target_col]].rename(columns={target_col: 'target'})

    # 👉 fit
    scaler = MinMaxScaler()
    scaler.fit(train_df_copy)

    # 👉 transform (np.ndarray 반환됨)
    train_scaled_vals = scaler.transform(train_df_copy)
    valid_scaled_vals = scaler.transform(valid_df_copy)
    test_scaled_vals  = scaler.transform(test_df_copy)

    # 👉 컬럼명 구성: features + 'target'
    scaled_columns = features + ['target']

    # 👉 DataFrame으로 복구
    train_scaled_df = pd.DataFrame(train_scaled_vals, columns=scaled_columns, index=train_df.index)
    valid_scaled_df = pd.DataFrame(valid_scaled_vals, columns=scaled_columns, index=valid_df.index)
    test_scaled_df  = pd.DataFrame(test_scaled_vals,  columns=scaled_columns, index=test_df.index)

    # 🟡 'date' 컬럼 유지
    for df_orig, df_scaled in zip([train_df, valid_df, test_df], [train_scaled_df, valid_scaled_df, test_scaled_df]):
        if ("date" in df_orig.columns) and ('region' in df_orig.columns):
            df_scaled["date"] = df_orig["date"]
            df_scaled["region"] = df_orig["region"]

    return train_scaled_df, valid_scaled_df, test_scaled_df, scaler


In [7]:
def generate_time_features(dates: pd.Series) -> np.ndarray:
    """
    날짜 Series를 받아 시간 특성 (month, day, weekday, dayofyear) 생성
    """
    dates = pd.to_datetime(dates)

    features = np.stack([
        dates.dt.month / 12.0,
        dates.dt.day / 31.0,
        dates.dt.weekday / 6.0,
        dates.dt.dayofyear / 366.0
    ], axis=1)

    return features  # shape: [len(dates), 4]

In [8]:
# 📌 시계열 window 데이터셋 생성 함수
def create_transformer_dataset(df, feature_cols, target_col, region_col, date_col,
                            input_len=30, label_len=15, pred_len=15):
    """
    Informer 입력에 맞게 window 기반 데이터셋 생성
    반환: x_enc, x_dec, y, region_id, x_mark_enc, x_mark_dec
    """
    x_enc_all, x_dec_all, y_all, region_ids = [], [], [], []
    x_mark_enc_all, x_mark_dec_all = [], []

    total_window = input_len + pred_len
    for i in range(len(df) - total_window + 1):
        enc_window = df.iloc[i:i+input_len]
        dec_input = df.iloc[i+input_len-label_len:i+input_len+pred_len]
        y = df[target_col].iloc[i+input_len:i+input_len+pred_len].values

        # Feature 입력
        x_enc = enc_window[feature_cols].values
        x_dec = dec_input[feature_cols].copy().values
        x_dec[label_len:] = 0  # 미래 구간은 zero

        # 시간 마킹 입력
        x_mark_enc = generate_time_features(enc_window[date_col])
        x_mark_dec = generate_time_features(dec_input[date_col])

        # 저장
        x_enc_all.append(x_enc)
        x_dec_all.append(x_dec)
        y_all.append(y)
        region_ids.append(df[region_col].iloc[i])
        x_mark_enc_all.append(x_mark_enc)
        x_mark_dec_all.append(x_mark_dec)

    return (
        np.array(x_enc_all),         # [N, input_len, C]
        np.array(x_dec_all),         # [N, label_len+pred_len, C]
        np.array(y_all),             # [N, pred_len]
        np.array(region_ids),        # [N]
        np.array(x_mark_enc_all),    # [N, input_len, T]
        np.array(x_mark_dec_all)     # [N, label_len+pred_len, T]
    )

In [9]:
def build_transformer_datasets_from_split(
    train_df,
    valid_df,
    test_df,
    feature_cols,
    region_col,
    input_len=30,
    label_len=15,
    pred_len=5
):

    region2idx = {r: i for i, r in enumerate(pd.concat([train_df, valid_df, test_df])[region_col].unique())}
    region_scalers = {}

    data = {
        "train": {"x_enc": [], "x_dec": [], "y": [], "region": [], "x_mark_enc": [], "x_mark_dec": []},
        "val":   {"x_enc": [], "x_dec": [], "y": [], "region": [], "x_mark_enc": [], "x_mark_dec": []},
        "test":  {"x_enc": [], "x_dec": [], "y": [], "region": [], "x_mark_enc": [], "x_mark_dec": []}
    }

    df_train_all, df_val_all, df_test_all = [], [], []

    for region in region2idx.keys():
        region_id = region2idx[region]

        # 지역별 분리
        train_r = train_df[train_df[region_col] == region].sort_values("date").reset_index(drop=True)
        valid_r = valid_df[valid_df[region_col] == region].sort_values("date").reset_index(drop=True)
        test_r = test_df[test_df[region_col] == region].sort_values("date").reset_index(drop=True)

        # 스케일링: train/valid은 volume_cnt_clean 기준, test는 원본 volume_cnt 기준
        scaled_train, scaled_valid, scaled_test, scaler = scale_features_from_split(
            train_df=train_r,
            valid_df=valid_r,
            test_df=test_r,
            features=feature_cols,
            target_col="volume_cnt",
            target_clean_col="volume_cnt_scaled"
        )
        region_scalers[region] = scaler

        # 👉 target 컬럼명을 'volume_cnt_scaled'로 변경
        scaled_train = scaled_train.rename(columns={"target": "volume_cnt"})
        scaled_valid = scaled_valid.rename(columns={"target": "volume_cnt"})
        scaled_test  = scaled_test.rename(columns={"target": "volume_cnt"})  # test는 원래 컬럼명 유지

        df_train_all.append(scaled_train)
        df_val_all.append(scaled_valid)
        df_test_all.append(scaled_test)

        for split_name, df_split, target_col in [
            ("train", scaled_train, "volume_cnt"),
            ("val", scaled_valid, "volume_cnt"),
            ("test", scaled_test, "volume_cnt")
        ]:
            x_enc, x_dec, y, _, x_mark_enc, x_mark_dec = create_transformer_dataset(
                df_split,
                feature_cols=feature_cols,
                target_col=target_col,
                region_col=region_col,
                date_col="date",
                input_len=input_len,
                label_len=label_len,
                pred_len=pred_len
            )

            data[split_name]["x_enc"].append(x_enc)
            data[split_name]["x_dec"].append(x_dec)
            data[split_name]["y"].append(y)
            data[split_name]["x_mark_enc"].append(x_mark_enc)
            data[split_name]["x_mark_dec"].append(x_mark_dec)
            data[split_name]["region"].extend([region_id] * len(x_enc))

    # 병합
    for split in ["train", "val", "test"]:
        for key in ["x_enc", "x_dec", "y", "x_mark_enc", "x_mark_dec"]:
            data[split][key] = np.concatenate(data[split][key], axis=0)
        data[split]["region"] = np.array(data[split]["region"])

    origin_df_dic = {
        "train": pd.concat(df_train_all, ignore_index=True),
        "val": pd.concat(df_val_all, ignore_index=True),
        "test": pd.concat(df_test_all, ignore_index=True)
    }

    return data, region_scalers, region2idx, origin_df_dic


In [10]:
feature_cols = [
                  'avg_area_size', 'avg_sales', 'avg_floor_cnt',

                  # 'pod_index', 'abs_vdd_index', 'canceld_index', 'direct_trade_ratio', 'anomaly_abs_vdd_score',

                  'wolse_volume_cnt', 'jeonse_volume_cnt', 'wolse_avg_area_size',
                  'jeonse_avg_area_size', 'wolse_avg_deposit', 'jeonse_avg_deposit',
                  'wolse_avg_monthly', 'wolse_avg_floor_cnt', 'jeonse_avg_floor_cnt',

                  'townhouse_volume_cnt', 'townhouse_avg_area_size',
                  'townhouse_avg_sales', 'townhouse_avg_floor_cnt',
                  'townhouse_jeonse_volume_cnt', 'townhouse_wolse_avg_area_size',
                  'townhouse_jeonse_avg_area_size', 'townhouse_wolse_avg_floor_cnt',
                  'townhouse_jeonse_avg_floor_cnt', 'townhouse_wolse_avg_deposit',
                  'townhouse_jeonse_avg_deposit', 'townhouse_wolse_avg_monthly',

                  'familyhouse_volume_cnt', 'familyhouse_avg_area_size',
                  'familyhouse_avg_sales', 'familyhouse_wolse_volume_cnt',
                  'familyhouse_jeonse_volume_cnt', 'familyhouse_wolse_avg_area_size',
                  'familyhouse_jeonse_avg_area_size', 'familyhouse_wolse_avg_deposit',
                  'familyhouse_jeonse_avg_deposit', 'familyhouse_wolse_avg_monthly',

                  'officehouse_volume_cnt', 'officehouse_avg_area_size',
                  'officehouse_avg_sales', 'officehouse_avg_floor_cnt',
                  'officehouse_wolse_volume_cnt', 'officehouse_jeonse_volume_cnt',
                  'officehouse_wolse_avg_area_size', 'officehouse_jeonse_avg_area_size',
                  'officehouse_wolse_avg_floor_cnt', 'officehouse_jeonse_avg_floor_cnt',
                  'officehouse_wolse_avg_deposit', 'officehouse_jeonse_avg_deposit',
                  'officehouse_wolse_avg_monthly', 'townhouse_wolse_volume_cnt'
              ]

# is_anomaly_score_outlier, is_anomaly_score_outlier_25.0%, is_anomaly_score_outlier_15.0%, is_anomaly_score_outlier_5.0%

# train, valid set은 정상 라벨 데이터만 반영
anomaly_score_label_col = 'is_anomaly_score_outlier_mean_2sigma_win30'
data_dict, scaler, region2idx, origin_df_dic = build_transformer_datasets_from_split(
                                                      train_df=apt_volume_train_df[apt_volume_train_df[f'{anomaly_score_label_col}'] == 0],
                                                      valid_df=apt_volume_valid_df[apt_volume_train_df[f'{anomaly_score_label_col}'] == 0],
                                                      test_df=apt_volume_test_df,
                                                      feature_cols=feature_cols,
                                                      region_col='region',
                                                      input_len=30,
                                                      label_len=10,
                                                      pred_len=1
                                                  )

In [11]:
len(apt_volume_train_df), len(apt_volume_valid_df), len(apt_volume_test_df)

(20550, 3025, 5350)

In [12]:
len(origin_df_dic['train']), len(origin_df_dic['val']), len(origin_df_dic['test'])

(19371, 2848, 5350)

In [13]:
# 📌 TRAIN SET
X_train_enc = data_dict['train']['x_enc']
X_train_dec = data_dict['train']['x_dec']
y_train     = data_dict['train']['y']
region_train = data_dict['train']['region']
X_mark_enc_train = data_dict['train']['x_mark_enc']
X_mark_dec_train = data_dict['train']['x_mark_dec']

# 📌 VALIDATION SET
X_val_enc = data_dict['val']['x_enc']
X_val_dec = data_dict['val']['x_dec']
y_val     = data_dict['val']['y']
region_val = data_dict['val']['region']
X_mark_enc_val = data_dict['val']['x_mark_enc']
X_mark_dec_val = data_dict['val']['x_mark_dec']

# 📌 TEST SET
X_test_enc = data_dict['test']['x_enc']
X_test_dec = data_dict['test']['x_dec']
y_test     = data_dict['test']['y']
region_test = data_dict['test']['region']
X_mark_enc_test = data_dict['test']['x_mark_enc']
X_mark_dec_test = data_dict['test']['x_mark_dec']

# 4. Torch Dataset 구성

In [14]:
import torch
from torch.utils.data import Dataset

class TransformerRegionDataset(Dataset):
    def __init__(self, x_enc, x_dec, y, region_ids, x_mark_enc, x_mark_dec):
        """
        x_enc:        [N, input_len, num_features]
        x_dec:        [N, label_len + pred_len, num_features]
        y:            [N, pred_len]
        region_ids:   [N]
        x_mark_enc:   [N, input_len, time_feature_dim]
        x_mark_dec:   [N, label_len + pred_len, time_feature_dim]
        """
        self.x_enc = x_enc
        self.x_dec = x_dec
        self.y = y
        self.region_ids = region_ids
        self.x_mark_enc = x_mark_enc
        self.x_mark_dec = x_mark_dec

    def __len__(self):
        return len(self.x_enc)

    def __getitem__(self, idx):
        x_enc = torch.tensor(self.x_enc[idx], dtype=torch.float32)
        x_dec = torch.tensor(self.x_dec[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        region_id = torch.tensor(self.region_ids[idx], dtype=torch.long)
        x_mark_enc = torch.tensor(self.x_mark_enc[idx], dtype=torch.float32)
        x_mark_dec = torch.tensor(self.x_mark_dec[idx], dtype=torch.float32)

        return x_enc, x_mark_enc, x_dec, x_mark_dec, y, region_id


In [15]:
batch_size = 64

# ✅ Train
train_dataset = TransformerRegionDataset(
    X_train_enc, X_train_dec, y_train, region_train,
    X_mark_enc_train, X_mark_dec_train
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

# ✅ Validation
val_dataset = TransformerRegionDataset(
    X_val_enc, X_val_dec, y_val, region_val,
    X_mark_enc_val, X_mark_dec_val
)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ✅ Test
test_dataset = TransformerRegionDataset(
    X_test_enc, X_test_dec, y_test, region_test,
    X_mark_enc_test, X_mark_dec_test
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [16]:
for x_enc, x_mark_enc, x_dec, x_mark_dec, y, region_id in train_loader:

    print("x_enc:", x_enc.shape)       # [64, 30, feature_dim]
    print("x_mark_enc:", x_mark_enc.shape)  # [64]

    print("x_dec:", x_dec.shape)       # [64, 30, feature_dim]
    print("x_mark_dec:", x_mark_dec.shape)  # [64]

    print("y:", y.shape)               # [64,] or [64, pred_len]
    print("region_id:", region_id.shape)  # [64]
    break

x_enc: torch.Size([64, 30, 48])
x_mark_enc: torch.Size([64, 30, 4])
x_dec: torch.Size([64, 11, 48])
x_mark_dec: torch.Size([64, 11, 4])
y: torch.Size([64, 1])
region_id: torch.Size([64])


# 5. Transformer Model

In [17]:
import torch.nn as nn
import torch.nn.functional as F


class ConvLayer(nn.Module):
    def __init__(self, c_in):
        super(ConvLayer, self).__init__()
        self.downConv = nn.Conv1d(in_channels=c_in,
                                  out_channels=c_in,
                                  kernel_size=3,
                                  padding=2,
                                  padding_mode='circular')
        self.norm = nn.BatchNorm1d(c_in)
        self.activation = nn.ELU()
        self.maxPool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        x = self.downConv(x.permute(0, 2, 1))
        x = self.norm(x)
        x = self.activation(x)
        x = self.maxPool(x)
        x = x.transpose(1, 2)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, attention, d_model, d_ff=None, dropout=0.1, activation="relu"):
        super(EncoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, attn_mask=None):
        new_x, attn = self.attention(
            x, x, x,
            attn_mask=attn_mask
        )
        x = x + self.dropout(new_x)

        y = x = self.norm1(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm2(x + y), attn


class Encoder(nn.Module):
    def __init__(self, attn_layers, conv_layers=None, norm_layer=None):
        super(Encoder, self).__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.conv_layers = nn.ModuleList(conv_layers) if conv_layers is not None else None
        self.norm = norm_layer

    def forward(self, x, attn_mask=None):
        # x [B, L, D]
        attns = []
        if self.conv_layers is not None:
            for attn_layer, conv_layer in zip(self.attn_layers, self.conv_layers):
                x, attn = attn_layer(x, attn_mask=attn_mask)
                x = conv_layer(x)
                attns.append(attn)
            x, attn = self.attn_layers[-1](x)
            attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                x, attn = attn_layer(x, attn_mask=attn_mask)
                attns.append(attn)

        if self.norm is not None:
            x = self.norm(x)

        return x, attns


class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, d_model, d_ff=None,
                 dropout=0.1, activation="relu"):
        super(DecoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, cross, x_mask=None, cross_mask=None):
        x = x + self.dropout(self.self_attention(
            x, x, x,
            attn_mask=x_mask
        )[0])
        x = self.norm1(x)

        x = x + self.dropout(self.cross_attention(
            x, cross, cross,
            attn_mask=cross_mask
        )[0])

        y = x = self.norm2(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm3(x + y)


class Decoder(nn.Module):
    def __init__(self, layers, norm_layer=None, projection=None):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, x_mask=None, cross_mask=None):
        for layer in self.layers:
            x = layer(x, cross, x_mask=x_mask, cross_mask=cross_mask)

        if self.norm is not None:
            x = self.norm(x)

        if self.projection is not None:
            x = self.projection(x)
        return x

In [18]:
class TriangularCausalMask():
    def __init__(self, B, L, device="cpu"):
        mask_shape = [B, 1, L, L]
        with torch.no_grad():
            self._mask = torch.triu(torch.ones(mask_shape, dtype=torch.bool), diagonal=1).to(device)

    @property
    def mask(self):
        return self._mask


class ProbMask():
    def __init__(self, B, H, L, index, scores, device="cpu"):
        _mask = torch.ones(L, scores.shape[-1], dtype=torch.bool).to(device).triu(1)
        _mask_ex = _mask[None, None, :].expand(B, H, L, scores.shape[-1])
        indicator = _mask_ex[torch.arange(B)[:, None, None],
                    torch.arange(H)[None, :, None],
                    index, :].to(device)
        self._mask = indicator.view(scores.shape).to(device)

In [19]:
import torch
import torch.nn as nn

import numpy as np
from math import sqrt


class FullAttention(nn.Module):
    def __init__(self, mask_flag=True, factor=5, scale=None, attention_dropout=0.1, output_attention=False):
        super(FullAttention, self).__init__()
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def forward(self, queries, keys, values, attn_mask):
        B, L, H, E = queries.shape
        _, S, _, D = values.shape
        scale = self.scale or 1. / sqrt(E)

        scores = torch.einsum("blhe,bshe->bhls", queries, keys)

        if self.mask_flag:
            if attn_mask is None:
                attn_mask = TriangularCausalMask(B, L, device=queries.device)

            scores.masked_fill_(attn_mask.mask, -np.inf)

        A = self.dropout(torch.softmax(scale * scores, dim=-1))
        V = torch.einsum("bhls,bshd->blhd", A, values)

        if self.output_attention:
            return (V.contiguous(), A)
        else:
            return (V.contiguous(), None)


class AttentionLayer(nn.Module):
    def __init__(self, attention, d_model, n_heads, d_keys=None,
                 d_values=None):
        super(AttentionLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)
        d_values = d_values or (d_model // n_heads)

        self.inner_attention = attention
        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_model, d_keys * n_heads)
        self.value_projection = nn.Linear(d_model, d_values * n_heads)
        self.out_projection = nn.Linear(d_values * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values, attn_mask):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads

        queries = self.query_projection(queries).view(B, L, H, -1)
        keys = self.key_projection(keys).view(B, S, H, -1)
        values = self.value_projection(values).view(B, S, H, -1)

        out, attn = self.inner_attention(
            queries,
            keys,
            values,
            attn_mask
        )
        out = out.view(B, L, -1)

        return self.out_projection(out), attn

In [20]:
def compared_version(ver1, ver2):
    """
    :param ver1
    :param ver2
    :return: ver1< = >ver2 False/True
    """
    list1 = str(ver1).split(".")
    list2 = str(ver2).split(".")

    for i in range(len(list1)) if len(list1) < len(list2) else range(len(list2)):
        if int(list1[i]) == int(list2[i]):
            pass
        elif int(list1[i]) < int(list2[i]):
            return -1
        else:
            return 1

    if len(list1) == len(list2):
        return True
    elif len(list1) < len(list2):
        return False
    else:
        return True

class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe[:, :x.size(1)]


class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if compared_version(torch.__version__, '1.5.0') else 2
        self.tokenConv = nn.Conv1d(in_channels=c_in, out_channels=d_model,
                                   kernel_size=3, padding=padding, padding_mode='circular', bias=False)
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)
        return x


class FixedEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(FixedEmbedding, self).__init__()

        w = torch.zeros(c_in, d_model).float()
        w.require_grad = False

        position = torch.arange(0, c_in).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)).exp()

        w[:, 0::2] = torch.sin(position * div_term)
        w[:, 1::2] = torch.cos(position * div_term)

        self.emb = nn.Embedding(c_in, d_model)
        self.emb.weight = nn.Parameter(w, requires_grad=False)

    def forward(self, x):
        return self.emb(x).detach()


class TemporalEmbedding(nn.Module):
    def __init__(self, d_model, embed_type='fixed', freq='h'):
        super(TemporalEmbedding, self).__init__()

        minute_size = 4
        hour_size = 24
        weekday_size = 7
        day_size = 32
        month_size = 13

        Embed = FixedEmbedding if embed_type == 'fixed' else nn.Embedding
        if freq == 't':
            self.minute_embed = Embed(minute_size, d_model)
        self.hour_embed = Embed(hour_size, d_model)
        self.weekday_embed = Embed(weekday_size, d_model)
        self.day_embed = Embed(day_size, d_model)
        self.month_embed = Embed(month_size, d_model)

    def forward(self, x):
        x = x.long()

        minute_x = self.minute_embed(x[:, :, 4]) if hasattr(self, 'minute_embed') else 0.
        hour_x = self.hour_embed(x[:, :, 3])
        weekday_x = self.weekday_embed(x[:, :, 2])
        day_x = self.day_embed(x[:, :, 1])
        month_x = self.month_embed(x[:, :, 0])

        return hour_x + weekday_x + day_x + month_x + minute_x


class TimeFeatureEmbedding(nn.Module):
    def __init__(self, d_model, embed_type='timeF', freq='h'):
        super(TimeFeatureEmbedding, self).__init__()

        freq_map = {'h': 4, 't': 5, 's': 6, 'm': 1, 'a': 1, 'w': 2, 'd': 3, 'b': 3}
        d_inp = freq_map[freq]
        self.embed = nn.Linear(d_inp, d_model, bias=False)

    def forward(self, x):
        return self.embed(x)


class DataEmbedding(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.temporal_embedding = TemporalEmbedding(d_model=d_model, embed_type=embed_type,
                                                    freq=freq) if embed_type != 'timeF' else TimeFeatureEmbedding(
            d_model=d_model, embed_type=embed_type, freq=freq)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        x = self.value_embedding(x) + self.temporal_embedding(x_mark) + self.position_embedding(x)
        return self.dropout(x)


class DataEmbedding_wo_pos(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding_wo_pos, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.temporal_embedding = TemporalEmbedding(d_model=d_model, embed_type=embed_type,
                                                    freq=freq) if embed_type != 'timeF' else TimeFeatureEmbedding(
            d_model=d_model, embed_type=embed_type, freq=freq)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        x = self.value_embedding(x) + self.temporal_embedding(x_mark)
        return self.dropout(x)

In [21]:
class APTRegionTransformer(nn.Module):
    """
    Vanilla Transformer with O(L^2) complexity (configs 없이 개별 파라미터로 초기화)
    """
    def __init__(self,
                 enc_in, dec_in, c_out,
                 region_vocab_size=25, region_emb_dim=4,
                 d_model=512, n_heads=8, d_ff=512,
                 e_layers=3, d_layers=2,
                 dropout=0.1, activation='gelu',
                 embed='fixed', freq='d',
                 factor=5,
                 pred_len=15,
                 output_attention=False,
                 device=torch.device('cuda:0')):

        super(APTRegionTransformer, self).__init__()

        self.pred_len = pred_len
        self.output_attention = output_attention
        self.region_emb_dim = region_emb_dim

        # 🟡 지역 임베딩 레이어 추가
        self.region_embedding = nn.Embedding(region_vocab_size, region_emb_dim)

        # Embedding
        self.enc_embedding = DataEmbedding(enc_in + region_emb_dim, d_model, embed, freq, dropout)
        self.dec_embedding = DataEmbedding(dec_in + region_emb_dim, d_model, embed, freq, dropout)

        # Encoder
        self.encoder = Encoder(
            [
                EncoderLayer(
                    AttentionLayer(
                        FullAttention(False, factor, attention_dropout=dropout, output_attention=output_attention),
                        d_model, n_heads),
                    d_model,
                    d_ff,
                    dropout=dropout,
                    activation=activation
                )
                for _ in range(e_layers)
            ],
            norm_layer=torch.nn.LayerNorm(d_model)
        )

        # Decoder
        self.decoder = Decoder(
            [
                DecoderLayer(
                    AttentionLayer(
                        FullAttention(True, factor, attention_dropout=dropout, output_attention=False),
                        d_model, n_heads),
                    AttentionLayer(
                        FullAttention(False, factor, attention_dropout=dropout, output_attention=False),
                        d_model, n_heads),
                    d_model,
                    d_ff,
                    dropout=dropout,
                    activation=activation
                )
                for _ in range(d_layers)
            ],
            norm_layer=torch.nn.LayerNorm(d_model),
            projection=nn.Linear(d_model, c_out, bias=True)
        )

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, region_id,
                enc_self_mask=None, dec_self_mask=None, dec_enc_mask=None):

        # 🟡 지역 임베딩 벡터 획득
        region_emb = self.region_embedding(region_id)  # [B, region_emb_dim]
        region_emb = region_emb.unsqueeze(1).expand(-1, x_enc.shape[1], -1)  # [B, seq_len, region_emb_dim]

        # 🟡 지역 임베딩 벡터를 입력에 concat
        x_enc = torch.cat([x_enc, region_emb], dim=-1)
        x_dec = torch.cat([x_dec, region_emb[:, :x_dec.shape[1], :]], dim=-1)

        enc_out = self.enc_embedding(x_enc, x_mark_enc)
        enc_out, attns = self.encoder(enc_out, attn_mask=enc_self_mask)

        dec_out = self.dec_embedding(x_dec, x_mark_dec)
        dec_out = self.decoder(dec_out, enc_out, x_mask=dec_self_mask, cross_mask=dec_enc_mask)

        if self.output_attention:
            return dec_out[:, -self.pred_len:, :], attns
        else:
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]


# 6. 모델 훈련

In [22]:
class RMSELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super(RMSELoss, self).__init__()
        self.mse = nn.MSELoss()
        self.eps = eps  # 0으로 나누는 것 방지

    def forward(self, y_pred, y_true):
        return torch.sqrt(self.mse(y_pred, y_true) + self.eps)

class MAELoss(nn.Module):
    def __init__(self):
        super(MAELoss, self).__init__()

    def forward(self, y_pred, y_true):
        return torch.mean(torch.abs(y_true - y_pred))

# MAPE Loss (Mean Absolute Percentage Error)
class MAPELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super(MAPELoss, self).__init__()
        self.eps = eps

    def forward(self, y_pred, y_true):
        return torch.mean(torch.abs((y_true - y_pred) / (y_true + self.eps))) * 100

# SMAPE = 2 * abs(y_pred - y_true) / (abs(y_true) + abs(y_pred))
class SMAPELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super(SMAPELoss, self).__init__()
        self.eps = eps

    def forward(self, y_pred, y_true):
        numerator = torch.abs(y_true - y_pred)
        denominator = torch.abs(y_true) + torch.abs(y_pred) + self.eps
        return torch.mean(2 * numerator / denominator) * 100

In [23]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, mae_loss_fn, rmse_loss_fn):
    model.train()
    total_loss = 0
    total_rmse = 0
    total_mae = 0

    for x_enc, x_mark_enc, x_dec, x_mark_dec, y, region_id in dataloader:
        x_enc       = x_enc.to(device)
        x_mark_enc  = x_mark_enc.to(device)
        x_dec       = x_dec.to(device)
        x_mark_dec  = x_mark_dec.to(device)
        y           = y.to(device)
        region_id   = region_id.to(device)

        optimizer.zero_grad()
        output = model(x_enc, x_mark_enc, x_dec, x_mark_dec, region_id)  # [B, pred_len, 1]
        output = output.squeeze(-1)  # [B, pred_len]

        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_rmse += rmse_loss_fn(output, y).item()
        total_mae += mae_loss_fn(output, y).item()

    n = len(dataloader)
    return {
        'loss': total_loss / n,
        'rmse': total_rmse / n,
        'mae': total_mae / n
    }


In [24]:
@torch.no_grad()
def evaluate(model, dataloader, criterion, device, mae_loss_fn, rmse_loss_fn):
    model.eval()
    total_loss = 0
    total_rmse = 0
    total_mae = 0

    for x_enc, x_mark_enc, x_dec, x_mark_dec, y, region_id in dataloader:

        x_enc       = x_enc.to(device)
        x_mark_enc  = x_mark_enc.to(device)
        x_dec       = x_dec.to(device)
        x_mark_dec  = x_mark_dec.to(device)
        y           = y.to(device)
        region_id   = region_id.to(device)

        output = model(x_enc, x_mark_enc, x_dec, x_mark_dec, region_id)
        output = output.squeeze(-1)

        loss = criterion(output, y)

        total_loss += loss.item()
        total_rmse += rmse_loss_fn(output, y).item()
        total_mae += mae_loss_fn(output, y).item()

    n = len(dataloader)
    return {
        'loss': total_loss / n,
        'rmse': total_rmse / n,
        'mae': total_mae / n
    }


# 7. Test 구간 예측

In [25]:
import numpy as np
import pandas as pd
import torch

@torch.no_grad()
def predict_all_regions_by_split(
    model,
    device,
    region2idx: dict,
    region_scaler_dict: dict,
    train_loader,
    valid_loader,
    test_loader,
    train_period: tuple,
    valid_period: tuple,
    test_period: tuple,
    input_len: int = 30,
    label_len: int = 10,
    pred_len: int = 1
):
    """
    train/valid/test에 대해 예측 및 오차 결과 DataFrame 생성

    Returns:
        dict of DataFrames: {"train": df, "valid": df, "test": df}
    """
    model.eval()
    result_dict = {}

    loader_dict = {
        "train": (train_loader, train_period),
        "valid": (valid_loader, valid_period),
        "test": (test_loader, test_period)
    }

    region_id_to_name = {v: k for k, v in region2idx.items()}

    for split, (loader, (start_date_str, _)) in loader_dict.items():
        results = []
        start_date = pd.to_datetime(start_date_str)

        for region_name, region_id in region2idx.items():
            preds, actuals, errors = [], [], []
            scaler = region_scaler_dict[region_name]
            volume_index = scaler.n_features_in_ - 1

            for x_enc, x_mark_enc, x_dec, x_mark_dec, y, region_ids in loader:
                mask = (region_ids == region_id)
                if mask.sum() == 0:
                    continue

                x_enc = x_enc[mask].to(device)
                x_mark_enc = x_mark_enc[mask].to(device)
                x_dec = x_dec[mask].to(device)
                x_mark_dec = x_mark_dec[mask].to(device)
                y = y[mask].to(device)
                region_ids = region_ids[mask].to(device)

                pred = model(x_enc, x_mark_enc, x_dec, x_mark_dec, region_ids).squeeze(-1)

                for i in range(pred.shape[0]):
                    pred_1d = pred[i].cpu().numpy()
                    y_1d = y[i].cpu().numpy()

                    pred_full = np.zeros((pred_1d.shape[0], scaler.n_features_in_))
                    y_full = np.zeros((y_1d.shape[0], scaler.n_features_in_))

                    pred_full[:, volume_index] = pred_1d
                    y_full[:, volume_index] = y_1d

                    pred_inv = scaler.inverse_transform(pred_full)[:, volume_index]
                    y_inv = scaler.inverse_transform(y_full)[:, volume_index]
                    error = np.abs(pred_inv - y_inv)

                    preds.append(pred_inv)
                    actuals.append(y_inv)
                    errors.append(error)

            if len(errors) == 0:
                continue

            preds = np.vstack(preds).flatten()
            actuals = np.vstack(actuals).flatten()
            errors = np.vstack(errors).flatten()

            forecast_start_date = start_date + pd.Timedelta(days=input_len)
            forecast_dates = pd.date_range(start=forecast_start_date, periods=len(preds))

            df_region = pd.DataFrame({
                "region": region_name,
                "forecast_date": forecast_dates,
                "pred": preds,
                "actual": actuals,
                "error": errors
            })
            results.append(df_region)

        result_dict[split] = pd.concat(results, ignore_index=True)

    return result_dict


# 8. Window Sliding 동적 임계 값

In [26]:
from scipy import stats

def adaptive_threshold_detection_with_valid(
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    region_col: str = 'region',
    error_col: str = 'error',
    date_col: str = 'forecast_date',
    baseline_days: int = 30,
    new_window_size: int = 30,
    threshold_k: float = 2.0,
    ztest_alpha: float = 0.05
) -> pd.DataFrame:
    """
    valid set의 마지막 기간을 baseline으로 사용하고,
    test set에 대해 adaptive threshold 이상탐지 수행

    Returns:
        pd.DataFrame: test_df에 is_adaptive_anomaly, dynamic_threshold, adaptive_threshold 컬럼 추가
    """
    results = []

    for region in test_df[region_col].unique():
        test_region = test_df[test_df[region_col] == region].sort_values(date_col).reset_index(drop=True)
        valid_region = valid_df[valid_df[region_col] == region].sort_values(date_col)

        # ✅ baseline 구간: valid set 마지막 baseline_days 일
        baseline_region = valid_region.tail(baseline_days)
        if len(baseline_region) < baseline_days:
            print(f"[경고] {region}: baseline 구간 데이터 부족")
            continue

        baseline_errors = baseline_region[error_col].values
        mu_baseline = np.mean(baseline_errors)
        sigma_baseline = np.std(baseline_errors)
        threshold = mu_baseline + threshold_k * sigma_baseline

        errors = test_region[error_col].values
        n = len(errors)

        anomalies = []
        thresholds = []

        # ✅ 슬라이딩 윈도우
        for i in range(0, n, new_window_size):
            new_window = errors[i:i+new_window_size]

            if len(new_window) < 5:
                anomalies.extend([False] * len(new_window))
                thresholds.extend([threshold] * len(new_window))
                continue

            mu_new = np.mean(new_window)
            sigma_new = np.std(new_window)

            # z-test
            z_score = np.abs(mu_new - mu_baseline) / np.sqrt((sigma_new**2/len(new_window)) + (sigma_baseline**2/baseline_days))
            p_value = 2 * (1 - stats.norm.cdf(z_score))

            if p_value < ztest_alpha:
                # baseline 업데이트
                baseline_errors = new_window
                mu_baseline = mu_new
                sigma_baseline = sigma_new
                threshold = mu_baseline + threshold_k * sigma_baseline

            window_anomalies = new_window > threshold
            anomalies.extend(window_anomalies)
            thresholds.extend([threshold] * len(new_window))

        anomalies = anomalies[:n]
        thresholds = thresholds[:n]

        test_region['dynamic_threshold'] = thresholds
        test_region['adaptive_threshold'] = np.mean(thresholds)
        test_region['is_adaptive_anomaly'] = anomalies

        results.append(test_region)

    return pd.concat(results, ignore_index=True)


# 9. 반복 실험

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score
import torch

MAX_TRIALS = 50  # 최대 시도 횟수
TARGET_F1 = 0.5
trial = 0
best_f1 = -1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rmse_fn = RMSELoss()
mae_fn = MAELoss()

best_model_state = None
best_trial = -1
best_train_df, best_valid_df, best_test_df = None, None, None

apt_volume_test_df_origin = pd.read_csv('/content/drive/MyDrive/sogang_thesis_prj/dataset/final_test3/apt_sales_volume_daily_register_202111_202412_test_anomaly_score_label_combi3.csv')

while trial < MAX_TRIALS:
    trial += 1
    print(f"\n🚀 [Trial {trial}] Start Training...")

    # 모델 초기화 (반복마다 새로)
    model = APTRegionTransformer(
        enc_in=48,            # x_enc의 마지막 차원
        dec_in=48,            # x_dec의 마지막 차원
        c_out=1,             # y의 feature 수 (현재 y가 [64, 5]라면 c_out=1, 다변량이면 다르게)
        pred_len=1,           # y 또는 미래 예측 길이 (5일 예측)
        region_vocab_size=25,
        region_emb_dim=64,
        d_model=64,
        d_ff=256,
        dropout=0.1,

        e_layers=2,
        d_layers=2,
        n_heads=4
    )

    model = model.to(device)

    criterion = nn.MSELoss()
    lr_rate = 1e-4
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_rate)

    # ✅ 훈련
    for epoch in range(50):
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device, mae_fn, rmse_fn)
        val_metrics = evaluate(model, val_loader, criterion, device, mae_fn, rmse_fn)

        print(f"[Epoch {epoch+1}]")
        print(f"Train Loss: {train_metrics['loss']:.4f}, RMSE: {train_metrics['rmse']:.4f}, MAE: {train_metrics['mae']:.4f}")
        print(f"Val   Loss: {val_metrics['loss']:.4f}, RMSE: {val_metrics['rmse']:.4f}, MAE: {val_metrics['mae']:.4f}")

    # ✅ 예측
    result_df_dict = predict_all_regions_by_split(
        model=model,
        device=device,
        region2idx=region2idx,
        region_scaler_dict=scaler,
        train_loader=train_loader,
        valid_loader=val_loader,
        test_loader=test_loader,
        train_period=("2021-11-01", "2024-01-31"),
        valid_period=("2024-02-01", "2024-05-31"),
        test_period=("2024-06-01", "2024-12-31"),
        input_len=30,
        label_len=10,
        pred_len=1
    )

    train_result_df = result_df_dict["train"]
    valid_result_df = result_df_dict["valid"]
    test_result_df  = result_df_dict["test"]

    # ✅ Adaptive threshold 이상탐지 수행
    adaptive_result_df = adaptive_threshold_detection_with_valid(
        valid_df=result_df_dict["valid"],
        test_df=test_result_df,
        threshold_k=2.0,
        new_window_size=30,
        baseline_days=30
    )

    # ✅ 원본 테스트 데이터 병합
    apt_volume_test_df_origin['date'] = pd.to_datetime(apt_volume_test_df_origin['date'])
    adaptive_result_df['forecast_date'] = pd.to_datetime(adaptive_result_df['forecast_date'])
    adaptive_result_df_temp = apt_volume_test_df_origin.merge(
        adaptive_result_df,
        left_on=['region', 'date'],
        right_on=['region', 'forecast_date']
    )

    # ✅ 평가 지표 계산
    y_true = adaptive_result_df_temp[anomaly_score_label_col].astype(int)
    y_pred = adaptive_result_df_temp['is_adaptive_anomaly'].astype(int)

    precision = precision_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)

    print(f"[Trial {trial}] Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    # ✅ F1 만족 시 종료
    if f1 < TARGET_F1:
        print(f"\n✅ Target F1-score {f1:.4f} achieved at trial {trial}.")
        torch.save(model.state_dict(), f'apt_volume_pred_transformer_50_202111_202412_{anomaly_score_label_col}_index_delete_{trial}.pth')

        train_result_df.to_csv(f'apt_volume_pred_transformer_50_pred_result_train_202111_202412_{anomaly_score_label_col}_index_delete_{trial}.csv', index=False)
        valid_result_df.to_csv(f'apt_volume_pred_transformer_50_pred_result_valid_202111_202412_{anomaly_score_label_col}_index_delete_{trial}.csv', index=False)
        test_result_df.to_csv(f'apt_volume_pred_transformer_50_pred_result_test_202111_202412_{anomaly_score_label_col}_index_delete_{trial}.csv', index=False)

        break

    # ✅ F1 갱신 시 저장
    if f1 < best_f1:
        best_f1 = f1
        best_model_state = model.state_dict()
        best_trial = trial
        best_train_df = train_result_df.copy()
        best_valid_df = valid_result_df.copy()
        best_test_df  = test_result_df.copy()

else:
    print(f"\n❌ F1-score target not met within trial limit. Saving best model from trial {best_trial} with F1-score {best_f1:.4f}.")
    torch.save(best_model_state, f'apt_volume_pred_transformer_50_202111_202412_{anomaly_score_label_col}_index_delete_best_trial{best_trial}.pth')

    best_train_df.to_csv(f'apt_volume_pred_transformer_50_pred_result_train_202111_202412_{anomaly_score_label_col}_index_delete_best_trial{best_trial}.csv', index=False)
    best_valid_df.to_csv(f'apt_volume_pred_transformer_50_pred_result_valid_202111_202412_{anomaly_score_label_col}_index_delete_best_trial{best_trial}.csv', index=False)
    best_test_df.to_csv(f'apt_volume_pred_transformer_50_pred_result_test_202111_202412_{anomaly_score_label_col}_index_delete_best_trial{best_trial}.csv', index=False)


🚀 [Trial 1] Start Training...
[Epoch 1]
Train Loss: 0.0683, RMSE: 0.2503, MAE: 0.2048
Val   Loss: 0.1601, RMSE: 0.3880, MAE: 0.2901
[Epoch 2]
Train Loss: 0.0461, RMSE: 0.2047, MAE: 0.1671
Val   Loss: 0.1487, RMSE: 0.3752, MAE: 0.2784
[Epoch 3]
Train Loss: 0.0428, RMSE: 0.1965, MAE: 0.1615
Val   Loss: 0.1636, RMSE: 0.3940, MAE: 0.2942
[Epoch 4]
Train Loss: 0.0407, RMSE: 0.1910, MAE: 0.1569
Val   Loss: 0.1794, RMSE: 0.4132, MAE: 0.3115
[Epoch 5]
Train Loss: 0.0392, RMSE: 0.1870, MAE: 0.1539
Val   Loss: 0.1931, RMSE: 0.4290, MAE: 0.3264
[Epoch 6]
Train Loss: 0.0378, RMSE: 0.1834, MAE: 0.1510
Val   Loss: 0.1973, RMSE: 0.4337, MAE: 0.3308
[Epoch 7]
Train Loss: 0.0371, RMSE: 0.1815, MAE: 0.1495
Val   Loss: 0.2005, RMSE: 0.4373, MAE: 0.3343
[Epoch 8]
Train Loss: 0.0365, RMSE: 0.1802, MAE: 0.1484
Val   Loss: 0.2053, RMSE: 0.4426, MAE: 0.3396
[Epoch 9]
Train Loss: 0.0361, RMSE: 0.1789, MAE: 0.1471
Val   Loss: 0.2080, RMSE: 0.4456, MAE: 0.3425
[Epoch 10]
Train Loss: 0.0359, RMSE: 0.1785, MAE: 0

In [ ]:
valid_result_df = pd.read_csv('/content/apt_volume_pred_transformer_50_pred_result_valid_202111_202412_is_anomaly_score_outlier_mean_2sigma_win30_index_exist_best_trial43.csv')
test_result_df = pd.read_csv('/content/apt_volume_pred_transformer_50_pred_result_test_202111_202412_is_anomaly_score_outlier_mean_2sigma_win30_index_exist_best_trial43.csv')

In [28]:
# ✅ result_details_df: region, forecast_date, pred, actual, error 컬럼을 가진 DataFrame

adaptive_result_df = adaptive_threshold_detection_with_valid(
    valid_df=valid_result_df,
    test_df=test_result_df,
    baseline_days=30
)

adaptive_result_df

,region,forecast_date,pred,actual,error,dynamic_threshold,adaptive_threshold,is_adaptive_anomaly
0,강남구,2024-07-01,10.159219,19.000000,8.840781,24.147641,11.661759,False
1,강남구,2024-07-02,10.961831,15.000001,4.038170,24.147641,11.661759,False
2,강남구,2024-07-03,11.197284,16.000000,4.802716,24.147641,11.661759,False
3,강남구,2024-07-04,11.188709,16.000000,4.811291,24.147641,11.661759,False
4,강남구,2024-07-05,11.291162,22.000000,10.708838,24.147641,11.661759,False
...,...,...,...,...,...,...,...,...
4595,중랑구,2024-12-27,1.705950,2.000000,0.294050,4.587076,6.609077,False
4596,중랑구,2024-12-28,1.681004,10.000000,8.318996,4.587076,6.609077,False
4597,중랑구,2024-12-29,1.247730,1.000000,0.247730,4.587076,6.609077,False
4598,중랑구,2024-12-30,1.516948,4.000000,2.483052,4.587076,6.609077,False


In [29]:
apt_volume_test_df_origin['date'] = pd.to_datetime(apt_volume_test_df_origin['date'])

adaptive_result_df['forecast_date'] = pd.to_datetime(adaptive_result_df['forecast_date'])

adaptive_result_df_temp = apt_volume_test_df_origin.merge(adaptive_result_df, left_on=['region', 'date'], right_on=['region', 'forecast_date'])

adaptive_result_df_temp.groupby('is_adaptive_anomaly').agg(
    volume_cnt=('volume_cnt', 'sum'),
    cancel_cnt=('cancel_cnt', 'sum'),

    pod_index=('pod_index', 'mean'),
    canceld_index=('canceld_index', 'mean'),
    direct_trade_index=('direct_trade_ratio', 'mean'),
    abs_vdd_index=('abs_vdd_index', 'mean'),

    anomaly_score=('anomaly_abs_vdd_score', 'mean')
)

,volume_cnt,cancel_cnt,pod_index,canceld_index,direct_trade_index,abs_vdd_index,anomaly_score
is_adaptive_anomaly,,,,,,,
False,26190,1072,0.338183,0.032736,0.042126,0.494423,0.655388
True,3020,131,0.333684,0.054915,0.047836,1.875735,2.048178


In [30]:
# 최종 결과
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# 실제 값 (정답 라벨)
print(anomaly_score_label_col)
y_true = adaptive_result_df_temp[anomaly_score_label_col].astype(int)

# 예측값 (bool → int로 변환)
y_pred = adaptive_result_df_temp['is_adaptive_anomaly'].astype(int)

# 평가 지표 계산
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
accuracy  = accuracy_score(y_true, y_pred)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"Accuracy  : {accuracy:.4f}")


is_anomaly_score_outlier_mean_2sigma_win30
Precision : 0.5823
Recall    : 0.4126
F1-score  : 0.4829
Accuracy  : 0.9572


In [31]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import pandas as pd

def find_best_threshold_k_and_window(
    valid_df,
    test_df,
    apt_volume_test_df_origin,
    label_col='is_anomaly_score_outlier_5.0%',
    k_list=np.arange(1.0, 3.0, 0.1),
    window_list=np.arange(1, 31, 1)
):
    best_k = None
    best_window = None
    best_f1 = -1
    best_result_df = None

    best_precision = None
    best_recall = None
    best_accuracy = None

    for k in k_list:
        for win in window_list:
            result_df = adaptive_threshold_detection_with_valid(
                valid_df=valid_df,
                test_df=test_df,
                threshold_k=k,
                new_window_size=win
            )

            result_df['forecast_date'] = pd.to_datetime(result_df['forecast_date'])
            result_df = apt_volume_test_df_origin.merge(
                result_df,
                left_on=['region', 'date'],
                right_on=['region', 'forecast_date']
            )

            y_true = result_df[label_col].astype(int)
            y_pred = result_df['is_adaptive_anomaly'].astype(int)

            if y_true.sum() == 0 and y_pred.sum() == 0:
                f1 = 0.0
                precision = 0.0
                recall = 0.0
                accuracy = 1.0
            else:
                f1 = f1_score(y_true, y_pred, zero_division=0)
                precision = precision_score(y_true, y_pred, zero_division=0)
                recall = recall_score(y_true, y_pred, zero_division=0)
                accuracy = accuracy_score(y_true, y_pred)

            print(f"[k={k:.2f}, win={win}] → F1 = {f1:.4f}, Precision = {precision:.4f}, Recall = {recall:.4f}, Accuracy = {accuracy:.4f}")

            if f1 > best_f1:
                best_f1 = f1
                best_k = k
                best_window = win
                best_result_df = result_df.copy()

                best_precision = precision
                best_recall = recall
                best_accuracy = accuracy

    print(f"\n✅ Best k = {best_k}, window = {best_window}")
    print(f"   ↳ F1-score  = {best_f1:.4f}")
    print(f"   ↳ Precision = {best_precision:.4f}")
    print(f"   ↳ Recall    = {best_recall:.4f}")
    print(f"   ↳ Accuracy  = {best_accuracy:.4f}")

    return best_k, best_window, best_f1, best_precision, best_recall, best_accuracy, best_result_df


In [32]:
best_k, best_window, best_f1, best_precision, best_recall, best_accuracy, best_result_df = find_best_threshold_k_and_window(
    valid_df=valid_result_df,
    test_df=test_result_df,
    apt_volume_test_df_origin=apt_volume_test_df_origin,
    label_col=anomaly_score_label_col
)

[k=1.00, win=1] → F1 = 0.0000, Precision = 0.0000, Recall = 0.0000, Accuracy = 0.9515
[k=1.00, win=2] → F1 = 0.0000, Precision = 0.0000, Recall = 0.0000, Accuracy = 0.9515
[k=1.00, win=3] → F1 = 0.0000, Precision = 0.0000, Recall = 0.0000, Accuracy = 0.9515
[k=1.00, win=4] → F1 = 0.0000, Precision = 0.0000, Recall = 0.0000, Accuracy = 0.9515
[k=1.00, win=5] → F1 = 0.2564, Precision = 0.1526, Recall = 0.8027, Accuracy = 0.7743
[k=1.00, win=6] → F1 = 0.2771, Precision = 0.1684, Recall = 0.7803, Accuracy = 0.8026
[k=1.00, win=7] → F1 = 0.2833, Precision = 0.1749, Recall = 0.7444, Accuracy = 0.8174
[k=1.00, win=8] → F1 = 0.2996, Precision = 0.1864, Recall = 0.7623, Accuracy = 0.8272
[k=1.00, win=9] → F1 = 0.2898, Precision = 0.1807, Recall = 0.7309, Accuracy = 0.8263
[k=1.00, win=10] → F1 = 0.3089, Precision = 0.1968, Recall = 0.7175, Accuracy = 0.8443
[k=1.00, win=11] → F1 = 0.3350, Precision = 0.2161, Recall = 0.7444, Accuracy = 0.8567
[k=1.00, win=12] → F1 = 0.3484, Precision = 0.2330, 